## Causal Inference Alignment Tutorial

### step 0 - setup
For first-time users, make sure conda is installed and updated.

For first-time users, create a conda environment to work from. \
For a lighter-weight environment for alignment only, run in the terminal 

```conda env create --file ../utils/firerx_ci.yml``` 

This will create a conda environment named `firerx_ci` that can be activated with 

```conda activate firerx_ci```

For the full environment (including machine learning packages), run 

```conda env create --file ../utils/firerx_env_v1.yaml``` 

This will create a conda environment named `firerx` that can be activated with 

```conda activate firerx```

Note: these environments use the conda installation of gdal which can have problems with requirements (such as poppler). This can be fixed by pip installing poppler-utils to get the correct version, but some troubleshooting may be required

### step 1 - create a config file for this tutorial
Below, initialize a dict with default values. Edit this to suit your use case.

In [ ]:

ci_alignment_def = {"core": {"multiprocessing_mode": True},                             #parallelize alignment
                    "name": {"collection_name": "colorado_ci/",            #(CHANGE ME) #name of collection (name of directory to create)
                            "collection_directory_prefix": "../data/aligned_raster/",   #where to save collection
                            "trim_step_out": "trimmed/",                                #subdirectoy to save trimmed layers
                            "reproj_step_out": "reprojected/",                          #subdirectory to save trimmed -> reprojected rasters
                            "align_step_out": "aligned/",                               #subdirectory to save trimmed -> reprojected -> aligned rasters
                            "raster_src": "../data/raw_raster/"},                       #source file for raw data
                    "params": {"subdiv_meter_overlap": 1000,                            #overlap for region subdivision/batching
                            "subdiv_meter_slop": 10,                                    #extra slop buffer for region subdivision/batching
                            "subdiv_hv_split": (1, 1),                                  #how many horizontal, vertical regions to create
                            "res_check_tol": 0.001,                                     #tolerance for resolution double-check
                            "output_nodata": -99999,                                    #no-data value
                            "resampling_mode": "n_meter"},                              #which alignment sampling mode to use
                    "skip": {"guiding_load": False,                                     #
                            "subdivision": False,                                       #
                            "trim": False,                                              #
                            "extent_reproj": False,                                     #
                            "raster_reproj": False,                                     #
                            "resolution_check": False,                                  #
                            "alignment": False},                                        #
                    "data": {"guiding_layer_idx": 0,                                    #
                            "extent_epsg_override": False, ### need 3785 for california #california layer is buggy, so we need to use this for CAL
                            "exclude": [],                                              #
                            "extent_info": {"extent_src": "../data/extent/colorado/",   # (CHANGE ME) which extent directory to use
                                            "extent_name": "Colorado_State_Boundary"},  # (CHANGE ME) name of extent in extent diretory
                            "data_info": [{"loc": "colorado_wue_2022.tif", "name": "ECOSTRESS_WUE_22",  # (CHANGE ME)
                                            "base_res": 70, "output_res": 70, "resample_res": 10,
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "override_input_proj": False},
                                           {"loc": "conus_age_1km_albers.tif", "name": "TREEAGE_mean",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 1000, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": "5070"},#},

                                           {"loc": "forest_aboveground_carbon_flux_ED_GEDI.tif","name": "GEDI_AGB",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 1000, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},

                                           {"loc": "PRISM_tmax_30yr_normal_800mM5_annual_bil.bil",
                                            "name": "PRISM_TEMPMAX_30_mean",
                                            "input_func": "align_bil", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 800, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "PRISM_tmean_30yr_normal_800mM5_annual_bil.bil",
                                            "name": "PRISM_TEMPMEAN_30_mean",
                                            "input_func": "align_bil", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 800, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "PRISM_tmin_30yr_normal_800mM5_annual_bil.bil",
                                            "name": "PRISM_TEMPMIN_30_mean",
                                            "input_func": "align_bil", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 800, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "PRISM_vpdmax_30yr_normal_800mM5_annual_bil.bil",
                                            "name": "PRISM_VAPORMAX_30_mean",
                                            "input_func": "align_bil", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 800, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "PRISM_vpdmin_30yr_normal_800mM5_annual_bil.bil",
                                            "name": "PRISM_VAPORMIN_30_mean",
                                            "input_func": "align_bil", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 800, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "PRISM_ppt_30yr_normal_800mM4_annual_bil.bil",
                                            "name": "PRISM_PRECIP_30_mean",
                                            "input_func": "align_bil", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 800, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},

                                           {"loc": "LC22_EVC_230.tif", "name": "LANDFIRE_EVC_22_mean",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 30, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "LC22_EVH_230.tif", "name": "LANDFIRE_EVH_22_mean",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 30, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "LC22_EVT_230.tif", "name": "LANDFIRE_EVT_22_mode",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mode",
                                            "base_res": 30, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "LC22_FVH_230.tif", "name": "LANDFIRE_FVH_22_mean",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 30, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "LC22_CC_230.tif", "name": "LANDFIRE_CC_22_mean",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 30, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},

                                            ### below here we have non-conus layers (ESI, slope/aspect/srtm)
                                           {"loc": "colorado_esi_2022.tif", "name": "ECOSTRESS_ESI_22",
                                            "base_res": 70, "output_res": 70, "resample_res": 10,
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "override_input_proj": False},
                                           {"loc": "western_conus_aspect.tif", "name": "ASPECT_mean",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 30, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "western_consus_slope.tif", "name": "SLOPE_mean",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 30, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                           {"loc": "western_conus_elevation.tif", "name": "SRTM_01_mean",
                                            "input_func": "align_tif", "avg_func": "align_avg", "avg_method": "mean",
                                            "base_res": 30, "output_res": 70, "resample_res": 10,
                                            "override_input_proj": False},
                                        ]}}

Once you have edited the config dict above, edit the name of the config file in the code below and run the cell to save it:

Note that the file name and directory structure is somewhat strict to prevent confusion between configs for different processes. So for alignment, the file name must be manage_data/configs/dbci_`<filename>`.json

In [ ]:
import json

YOUR_FILE_NAME = "config_tutorial"
with open("configs/dbci_" + YOUR_FILE_NAME + ".json", "w") as config_out:
        json.dump(ci_alignment_def, config_out)

## step 2 - align the files

In the terminal, go to the manage_data directory. Run alignment with the following command after replacing the file name you chose above:

``` python batch_align_raster.py config=<YOUR FILE NAME> ```

Note that the file name you provide should not include the beginning of the file path or the extension

## step 3 - wait and be amazed